# Distance Metrics Analysis (ETL Pipeline)

Extracts, transforms, and analyzes distance metric tables (`*distance_metrics_*.csv`) produced by `notebooks/single_region_full_validation.ipynb` across multiple single-region validation runs.

### Pipeline Highlights:
- Configurable input paths to target validation execution folders.
- Dynamic extraction of kinematic region `(ET, ETA)` from filenames.
- Lineage tracking of origin `ExecutionFolder`.
- Consolidated multi-region analysis and export.

In [21]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as mh

mh.style.use("ATLAS")

from src.core.Validation.DistanceMetricsETL import DistanceMetricsETL

print(f"Project root: {PROJECT_ROOT}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: /eos/home-i04/p/pmourafr/egamma_master


## 1. ETL Configuration

Specify the validation folders containing the `Tables/` directory, or provide folder paths directly.

In [22]:
VALIDATION_FOLDERS: list[Path] = [
    PROJECT_ROOT / "notebooks/SingleRegionValidation_config_nameModelV9_HighBatch_100Rings_id20261029001623",
    PROJECT_ROOT / "notebooks/SingleRegionValidation_config_nameModelV10_HighBatch_100Rings_id20261029043936",
]

# Auto-discover all SingleRegionValidation folders if list is empty
if not VALIDATION_FOLDERS:
    discovered = sorted(list((PROJECT_ROOT / "notebooks").glob("SingleRegionValidation_*")))
    VALIDATION_FOLDERS = [d for d in discovered if d.is_dir()]

FILE_PATTERN: str = "*distance_metrics_*.csv"

print(f"Configured {len(VALIDATION_FOLDERS)} folders to scan:")
for folder in VALIDATION_FOLDERS:
    print(f" - {folder}")

Configured 2 folders to scan:
 - /eos/home-i04/p/pmourafr/egamma_master/notebooks/SingleRegionValidation_config_nameModelV9_HighBatch_100Rings_id20261029001623
 - /eos/home-i04/p/pmourafr/egamma_master/notebooks/SingleRegionValidation_config_nameModelV10_HighBatch_100Rings_id20261029043936


## 2. Execute Extraction & Consolidation

In [38]:
etl = DistanceMetricsETL(folder_paths=VALIDATION_FOLDERS, file_pattern=FILE_PATTERN)
print(f"Discovered {len(etl.discovered_files)} distance metric CSV files.")

df = etl.extract_and_consolidate()
print(f"Consolidated DataFrame shape: {df.shape}")
display(df[
    (df["Feature"]=="Eratio") &
    (~df["Reference"].str.contains('Athena')) &
    (~df["Strategy"].str.contains('Athena'))
    ])

Discovered 12 distance metric CSV files.
Consolidated DataFrame shape: (336, 20)


,Feature,Reference,Strategy,Comparison,Ref Count,Selected Count,KS Distance (D),KS Uncertainty,Wasserstein Distance,Wasserstein Uncertainty,KL Divergence,KL Uncertainty,JS Divergence,JS Uncertainty,p-value,Distribution Equality,ET,ETA,ExecutionFolder,MetricType
22,Eratio,Model V9 (50 Rings),Model V9 (100 Rings),Model V9 (100 Rings) vs Model V9 (50 Rings),4825,4885,0.008336,0.005644,0.003483,0.002906,0.001018,0.002153,0.000370,0.000782,9.952457e-01,Consistent (p ≥ 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,model1_vs_model2_background
46,Eratio,Model V9 (50 Rings),Model V9 (100 Rings),Model V9 (100 Rings) vs Model V9 (50 Rings),3764,3764,0.001594,0.005839,0.000578,0.002569,0.000337,0.002186,0.000121,0.000749,1.000000e+00,Consistent (p ≥ 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,model1_vs_model2_signal
129,Eratio,True Background,Model V9 (100 Rings),Model V9 (100 Rings) vs True Background,10384,4885,0.233209,0.008621,0.155185,0.005474,0.140581,0.010406,0.048428,0.003412,2.286649e-159,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,vs_true_background
131,Eratio,True Background,Model V9 (50 Rings),Model V9 (50 Rings) vs True Background,10384,4825,0.236620,0.008449,0.156434,0.005397,0.141481,0.009555,0.049011,0.003163,9.888429e-163,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,vs_true_background
165,Eratio,True Signal,Model V9 (100 Rings),Model V9 (100 Rings) vs True Signal,3994,3764,0.040645,0.007972,0.027006,0.005017,0.008171,0.003880,0.002763,0.001211,3.190618e-03,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,vs_true_signal
167,Eratio,True Signal,Model V9 (50 Rings),Model V9 (50 Rings) vs True Signal,3994,3764,0.040911,0.008956,0.026714,0.006028,0.007662,0.004220,0.002601,0.001294,2.932814e-03,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,vs_true_signal
190,Eratio,Model V10 (50 Rings),Model V10 (100 Rings),Model V10 (100 Rings) vs Model V10 (50 Rings),5313,5068,0.014079,0.006904,0.008522,0.004298,0.001073,0.001738,0.000383,0.000617,6.751526e-01,Consistent (p ≥ 0.05),1,1,SingleRegionValidation_config_nameModelV10_Hig...,model1_vs_model2_background
214,Eratio,Model V10 (50 Rings),Model V10 (100 Rings),Model V10 (100 Rings) vs Model V10 (50 Rings),3764,3764,0.001594,0.006085,0.000584,0.002751,0.000404,0.002468,0.000145,0.000858,1.000000e+00,Consistent (p ≥ 0.05),1,1,SingleRegionValidation_config_nameModelV10_Hig...,model1_vs_model2_signal
297,Eratio,True Background,Model V10 (100 Rings),Model V10 (100 Rings) vs True Background,10384,5068,0.208065,0.008382,0.139242,0.004756,0.110367,0.007995,0.038525,0.002693,8.201475e-130,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV10_Hig...,vs_true_background
299,Eratio,True Background,Model V10 (50 Rings),Model V10 (50 Rings) vs True Background,10384,5313,0.195078,0.009129,0.130725,0.005415,0.096458,0.008461,0.033853,0.002911,1.151479e-117,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV10_Hig...,vs_true_background


## 3. Executive Summary by Region & Run

In [24]:
summary_df = etl.get_summary(df)
if not summary_df.empty:
    display(summary_df)
else:
    print("No records found to summarize.")

,ExecutionFolder,ET,ETA,MetricType,RecordCount
0,SingleRegionValidation_config_nameModelV10_Hig...,1,1,model1_vs_model2_background,24
1,SingleRegionValidation_config_nameModelV10_Hig...,1,1,model1_vs_model2_signal,24
2,SingleRegionValidation_config_nameModelV10_Hig...,1,1,vs_athena_cut_background,24
3,SingleRegionValidation_config_nameModelV10_Hig...,1,1,vs_athena_cut_signal,24
4,SingleRegionValidation_config_nameModelV10_Hig...,1,1,vs_true_background,36
5,SingleRegionValidation_config_nameModelV10_Hig...,1,1,vs_true_signal,36
6,SingleRegionValidation_config_nameModelV9_High...,1,1,model1_vs_model2_background,24
7,SingleRegionValidation_config_nameModelV9_High...,1,1,model1_vs_model2_signal,24
8,SingleRegionValidation_config_nameModelV9_High...,1,1,vs_athena_cut_background,24
9,SingleRegionValidation_config_nameModelV9_High...,1,1,vs_athena_cut_signal,24


## 4. Query & Inspect Specific Metrics & Features

In [25]:
if not df.empty:
    print("Available Metric Types:", df["MetricType"].unique())
    print("Available Features:", df["Feature"].unique() if "Feature" in df.columns else "N/A")

    # Example pivot / inspection
    metric_type_target = df["MetricType"].unique()[0]
    subset = df[df["MetricType"] == metric_type_target]
    print(f"\nSample data for {metric_type_target}:")
    display(subset.head())
else:
    print("Consolidated DataFrame is empty.")

Available Metric Types: ['model1_vs_model2_background' 'model1_vs_model2_signal'
 'vs_athena_cut_background' 'vs_athena_cut_signal' 'vs_true_background'
 'vs_true_signal']
Available Features: ['trig_L2_calo_et' 'trig_L2_calo_eta' 'trig_L2_calo_phi'
 'trig_L2_calo_e237' 'trig_L2_calo_e277' 'trig_L2_calo_fracs1'
 'trig_L2_calo_weta2' 'trig_L2_calo_ehad1' 'trig_L2_calo_emaxs1'
 'trig_L2_calo_e2tsts1' 'trig_L2_calo_wstot' 'Eratio']

Sample data for model1_vs_model2_background:


,Feature,Reference,Strategy,Comparison,Ref Count,Selected Count,KS Distance (D),KS Uncertainty,Wasserstein Distance,Wasserstein Uncertainty,KL Divergence,KL Uncertainty,JS Divergence,JS Uncertainty,p-value,Distribution Equality,ET,ETA,ExecutionFolder,MetricType
0,trig_L2_calo_et,Model V9 (50 Rings),Model V9 (100 Rings),Model V9 (100 Rings) vs Model V9 (50 Rings),4825,4885,0.004155,0.005358,16.628878,23.797069,0.000802,0.001884,0.000290,0.000687,1.000000,Consistent (p ≥ 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,model1_vs_model2_background
1,trig_L2_calo_et,Model V9 (50 Rings),Athena Loose Cut,Athena Loose Cut vs Model V9 (50 Rings),4825,4309,0.033937,0.009245,173.888109,47.986564,0.007913,0.003707,0.002733,0.001260,0.010246,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,model1_vs_model2_background
2,trig_L2_calo_eta,Model V9 (50 Rings),Model V9 (100 Rings),Model V9 (100 Rings) vs Model V9 (50 Rings),4825,4885,0.006995,0.006027,0.010359,0.012521,0.000536,0.001371,0.000194,0.000495,0.999707,Consistent (p ≥ 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,model1_vs_model2_background
3,trig_L2_calo_eta,Model V9 (50 Rings),Athena Loose Cut,Athena Loose Cut vs Model V9 (50 Rings),4825,4309,0.053555,0.007323,0.046787,0.016384,0.043599,0.006823,0.015827,0.002439,0.000004,Distinct (p < 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,model1_vs_model2_background
4,trig_L2_calo_phi,Model V9 (50 Rings),Model V9 (100 Rings),Model V9 (100 Rings) vs Model V9 (50 Rings),4825,4885,0.005269,0.004619,0.010745,0.013263,0.000704,0.001745,0.000254,0.000623,1.000000,Consistent (p ≥ 0.05),1,1,SingleRegionValidation_config_nameModelV9_High...,model1_vs_model2_background
